<a href="https://colab.research.google.com/github/ust-buan-640/buan-640-notebooks/blob/main/notebooks/m10-0-homes-anova-preview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data of Homes: Comparing More Than Two Groups

In class today we go back to the listings you collected in the Data of Homes exercise. In Module 9 we compared two groups. Today we ask what changes when there are more than two groups, whether we can trust the answer, and whether the type of a home is what drives its price.

Run the cells in order, from top to bottom, with **Shift + Enter**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
url = "https://raw.githubusercontent.com/ust-buan-640/buan-640-notebooks/main/data/data_of_homes_f26.csv"
df = pd.read_csv(url)
df[["zip", "property_type", "list_price", "sqft", "beds"]].head()

## 1. Warm-up: two groups

Are single-family homes more expensive than condos? This is a Module 9 question: a comparison of two means.

In [ ]:
df.groupby("property_type")["list_price"].agg(["count", "mean", "std"]).round(0)

In [ ]:
sf = df.loc[df["property_type"] == "Single Family", "list_price"]
condo = df.loc[df["property_type"] == "Condo", "list_price"]

t_stat, p_value = stats.ttest_ind(sf, condo, equal_var=False)
print(f"Difference in sample means: {sf.mean() - condo.mean():,.0f}")
print(f"t = {t_stat:.2f}, p-value = {p_value:.2e}")

**Question to you:** what do you conclude? Write the conclusion in the language of the problem.

## 2. Three groups

Our data have three property types. The question becomes: does the mean list price differ among single-family homes, townhouses, and condos?

**Question to you:** we already know how to compare two groups. Why not simply run a t-test for every pair of types?

In [ ]:
order = ["Condo", "Townhouse", "Single Family"]
means = df.groupby("property_type")["list_price"].mean()

plt.figure(figsize=(8, 4))
for i, t in enumerate(order):
    prices = df.loc[df["property_type"] == t, "list_price"]
    plt.scatter(np.full(len(prices), i) + np.random.uniform(-0.08, 0.08, len(prices)),
                prices, alpha=0.5, color="gray")
    plt.scatter(i, means[t], color="black", marker="D", s=100)
plt.xticks(range(3), order)
plt.ylabel("List price, USD")
plt.title("List prices by property type (diamonds are the group means)")
plt.ticklabel_format(style="plain", axis="y")
plt.show()

Look at two kinds of variation in this plot. The group means differ from each other: this is the variation **between** groups. The homes inside each group also differ from each other: this is the variation **within** groups. ANOVA (analysis of variance) asks whether the group means are further apart than we would expect from the variation within the groups alone.

In [ ]:
model = ols("list_price ~ C(property_type)", data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

**Question to you:** find the F statistic and the p-value. What are the null and alternative hypotheses, and what do you conclude at α = 0.05?

The ANOVA tells us that at least two population means differ, but not which ones. For that, we use a post hoc procedure. Tukey's HSD compares every pair and adjusts for the number of comparisons.

In [ ]:
tukey = pairwise_tukeyhsd(df["list_price"], df["property_type"])
print(tukey.summary())

**Question to you:** which pairs of property types differ? Which pair cannot be told apart?

## 3. Can we trust this p-value?

ANOVA rests on three assumptions: the observations are independent, the response is approximately normal within each group, and the groups have similar variances. Independence comes first, and we check it against how the data were collected.

Recall the design: seven zip codes were drawn at random, and then listings were sampled within each zip.

In [ ]:
zip_means = df.groupby("zip")["list_price"].mean()

plt.figure(figsize=(9, 4))
for i, (z, grp) in enumerate(df.groupby("zip")):
    plt.scatter([i] * len(grp), grp["list_price"], alpha=0.4, color="gray")
    plt.scatter(i, zip_means[z], color="black", marker="D", s=100)
plt.xticks(range(len(zip_means)), zip_means.index)
plt.xlabel("Zip code")
plt.ylabel("List price, USD")
plt.title("List prices by zip code (diamonds are the zip means)")
plt.ticklabel_format(style="plain", axis="y")
plt.show()

In [ ]:
pd.crosstab(df["zip"], df["property_type"])

**Question to you:** homes in the same zip resemble each other, and some zips contain mostly one property type. What does this mean for the assumption that the 93 homes are 93 independent observations? Is the evidence in our ANOVA as strong as the p-value suggests?

Now the other two assumptions. Compare the standard deviations of the three groups, and look at the shape of the price distribution.

In [ ]:
df.groupby("property_type")["list_price"].std().round(0)

In [ ]:
plt.hist(df["list_price"], bins=30, edgecolor="black")
plt.xlabel("List price, USD")
plt.ylabel("# of homes")
plt.ticklabel_format(style="plain", axis="x")
plt.xticks(rotation=45)
plt.show()

Prices are strongly right-skewed, and the spread among single-family homes is more than three times the spread among condos. A common remedy is to analyze the logarithm of price, which compresses the long right tail.

In [ ]:
df["log_price"] = np.log(df["list_price"])
print(df.groupby("property_type")["log_price"].std().round(3))
sm.stats.anova_lm(ols("log_price ~ C(property_type)", data=df).fit(), typ=2)

## 4. Does the type of a home drive its price?

**Question to you:** why are single-family homes more expensive than condos? Make a guess before you run the next cell.

In [ ]:
df["price_per_sqft"] = df["list_price"] / df["sqft"]
df.groupby("property_type")[["sqft", "list_price", "price_per_sqft"]].mean().round(0)

Single-family homes are much larger on average, while the price per square foot is similar across the three types. Let us look at price against size. This is a preview of Module 11, where we fit a line to data like these.

In [ ]:
colors = {"Condo": "tab:blue", "Townhouse": "tab:orange", "Single Family": "tab:green"}
plt.figure(figsize=(8, 5))
for t, grp in df.groupby("property_type"):
    plt.scatter(grp["sqft"], grp["list_price"], alpha=0.6, color=colors[t], label=t)

line = ols("list_price ~ sqft", data=df).fit()
xs = np.linspace(df["sqft"].min(), df["sqft"].max(), 100)
plt.plot(xs, line.params["Intercept"] + line.params["sqft"] * xs, color="black")
plt.xlabel("Living area, sq ft")
plt.ylabel("List price, USD")
plt.ticklabel_format(style="plain", axis="y")
plt.legend()
plt.show()

print(f"Estimated slope: {line.params['sqft']:.1f} USD per square foot")
print(f"R-squared: {line.rsquared:.3f}")

**Question to you:** how would you interpret the slope in words? What does R-squared tell us?

Finally, we put both size and property type into one model. This is a preview of Module 12. The coefficients of the property types now compare homes **of the same size**.

In [ ]:
both = ols("list_price ~ sqft + C(property_type)", data=df).fit()
print(both.summary().tables[1])

In [ ]:
print(f"Single-family vs. condo, ignoring size:       {sf.mean() - condo.mean():,.0f}")
print(f"Single-family vs. condo, holding size constant: {both.params['C(property_type)[T.Single Family]']:,.0f}")

**Question to you:** the single-family premium was more than $360,000. Once we hold size constant, it is close to zero and not significant. Which of these two numbers answers the question "how much more does a typical single-family listing cost?", and which answers "is a single-family home more expensive than a condo of the same size?"

**Question to you:** could we run an experiment in which property type is randomly assigned to homes? If not, what kind of experiment could a real estate agency run?